In [2]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/rishh19/FlyRank-AI-Internship"
REPO_DIR = "FlyRank-AI-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True,
        )
    os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/FlyRank-AI-Internship


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [4]:
!pip -q install duckdb huggingface_hub pandas pyarrow

In [5]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [6]:
import duckdb

con = duckdb.connect()

con.sql(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My baseline rule prioritizes pages that have high search impressions but relatively low clicks or poor average search position. These pages may represent opportunities for content refresh and SEO improvement.

**Reason Codes**
- HIGH_IMPRESSIONS_LOW_CLICKS
- LOW_VISIBILITY
- REVIEW_CONTENT

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_reason_codes = {
    "HIGH_IMPRESSIONS_LOW_CLICKS": "High impressions but low clicks",
    "LOW_VISIBILITY": "Poor average search position",
    "REVIEW_CONTENT": "Needs manual review"
}

print("Reason Codes:")
for code, description in rule_reason_codes.items():
    print(f"{code}: {description}")

Reason Codes:
HIGH_IMPRESSIONS_LOW_CLICKS: High impressions but low clicks
LOW_VISIBILITY: Poor average search position
REVIEW_CONTENT: Needs manual review


## 2. Build the ranked queue (writes the CSV)
I built a simple baseline scoring rule using three observable Google Search Console signals:

- Higher impressions increase the score.
- Lower clicks increase the priority.
- Poor average position increases the priority.

The rule assigns a score, a reason code, and an action label. The ranked queue is then saved as `work/outputs/baseline_action_score.csv`.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

query = f"""
COPY (

SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    (
        gsc_impressions * 0.01
        - gsc_clicks * 0.20
        + gsc_avg_position
    ) AS baseline_score,

    CASE
        WHEN gsc_impressions > 1000 AND gsc_clicks < 100
            THEN 'HIGH_IMPRESSIONS_LOW_CLICKS'
        WHEN gsc_avg_position > 20
            THEN 'LOW_VISIBILITY'
        ELSE 'REVIEW_CONTENT'
    END AS reason_code,

    'Review Content' AS action

FROM read_parquet('{path}')

ORDER BY baseline_score DESC

) TO 'work/outputs/baseline_action_score.csv'
WITH (HEADER, DELIMITER ',');
"""

con.execute(query)

print("CSV created successfully!")

top10 = con.sql("""
SELECT *
FROM read_csv_auto('work/outputs/baseline_action_score.csv')
LIMIT 10
""").df()

top10

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV created successfully!


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,2026-03-14,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,1,0,498.0,498.01,LOW_VISIBILITY,Review Content
1,2026-03-22,client_23a62021009f63c4,content_13ef8874a9a1ef5e,1,0,497.0,497.01,LOW_VISIBILITY,Review Content
2,2026-03-30,client_23a62021009f63c4,content_aacb637aa920b8ea,1,0,495.0,495.01,LOW_VISIBILITY,Review Content
3,2026-03-23,client_23a62021009f63c4,content_c7ebdf81f488f0d8,1,0,480.0,480.01,LOW_VISIBILITY,Review Content
4,2026-03-03,client_20259bd6705d81d4,content_8c34799f566ce23c,1,0,469.0,469.01,LOW_VISIBILITY,Review Content
5,2026-03-22,client_23a62021009f63c4,content_6f6a949b6e7eb3ae,1,0,465.0,465.01,LOW_VISIBILITY,Review Content
6,2026-03-03,client_e547b89c05043229,content_aa376cef98a5fae8,1,0,447.0,447.01,LOW_VISIBILITY,Review Content
7,2026-03-22,client_23a62021009f63c4,content_070944208ef3a890,1,0,445.0,445.01,LOW_VISIBILITY,Review Content
8,2026-03-31,client_23a62021009f63c4,content_74e1dddeda79c23c,1,0,444.0,444.01,LOW_VISIBILITY,Review Content
9,2026-03-15,client_20259bd6705d81d4,content_9afdc38dbabc43c6,2,0,403.0,403.02,LOW_VISIBILITY,Review Content


## 3. Top-20 review

The following table shows the highest-priority pages identified by my baseline rule. These recommendations are intended for manual review before taking action.

For each page, I record:
- Action
- Reason code
- Confidence
- What could make the recommendation wrong

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = con.sql("""
SELECT *
FROM read_csv_auto('work/outputs/baseline_action_score.csv')
LIMIT 20
""").df()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonality, marketing campaigns, or temporary ranking changes."
)

top20[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

,content_hash_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,content_8390ee56e6ea98ee,498.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
1,content_13ef8874a9a1ef5e,497.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
2,content_aacb637aa920b8ea,495.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
3,content_c7ebdf81f488f0d8,480.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
4,content_8c34799f566ce23c,469.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
5,content_6f6a949b6e7eb3ae,465.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
6,content_aa376cef98a5fae8,447.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
7,content_070944208ef3a890,445.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
8,content_74e1dddeda79c23c,444.010000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."
9,content_9afdc38dbabc43c6,403.020000,LOW_VISIBILITY,Review Content,Medium,"Seasonality, marketing campaigns, or temporary..."


## 4. Weak picks + leakage check

Some recommendations may be incorrect because search performance can be affected by seasonality, algorithm updates, or marketing campaigns that are not represented in this dataset.

This baseline uses only current observable signals and does not use future information, labels, or FlyRank product flags. Therefore, no intentional data leakage is present.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")

print("✓ No future window data used.")
print("✓ No label-derived columns used.")
print("✓ No FlyRank product flags used.")

print("\nPossible weak picks:")
print("- Seasonal traffic changes")
print("- Temporary ranking fluctuations")
print("- External marketing campaigns")

Leakage Check
✓ No future window data used.
✓ No label-derived columns used.
✓ No FlyRank product flags used.

Possible weak picks:
- Seasonal traffic changes
- Temporary ranking fluctuations
- External marketing campaigns


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.